
# Inventory Planning and Replenishment Recommendations


In [2]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Define project folders
DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")

# Load our trained demand forecasting model
model_package = joblib.load(
    MODEL_DIR / "horizon_lightgbm.pkl"
)

forecast_model = model_package["model"]
feature_columns = model_package["feature_columns"]

print("Inventory planning environment ready!")
print("Forecasting model loaded successfully.")
print("Number of model features:", len(feature_columns))


Inventory planning environment ready!
Forecasting model loaded successfully.
Number of model features: 10



### 2. Create Simulated Inventory Data


In [3]:

# Load the product list from our original sales data
sales = pd.read_csv(
    DATA_DIR / "sales_train_validation.csv",
    usecols=["item_id", "dept_id", "cat_id", "store_id"]
)

# Keep products from our selected store
inventory = (
    sales[sales["store_id"] == "CA_1"]
    .copy()
    .reset_index(drop=True)
)

# Create reproducible simulated inventory values
rng = np.random.default_rng(42)

inventory["current_stock"] = rng.integers(
    10, 101, size=len(inventory)
)

inventory["incoming_stock"] = rng.integers(
    0, 31, size=len(inventory)
)

inventory["lead_time_days"] = rng.integers(
    3, 15, size=len(inventory)
)

inventory["safety_stock"] = rng.integers(
    5, 26, size=len(inventory)
)

print("Inventory shape:", inventory.shape)
print("Unique products:", inventory["item_id"].nunique())
print("Missing values:", inventory.isna().sum().sum())

display(inventory.head())


Inventory shape: (3049, 8)
Unique products: 3049
Missing values: 0


,item_id,dept_id,cat_id,store_id,current_stock,incoming_stock,lead_time_days,safety_stock
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,18,21,12,12
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,80,25,5,14
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,69,5,5,22
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,49,23,13,22
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,49,11,8,9



### 3. Calculate Inventory Position


In [4]:

# Calculate inventory position
inventory["inventory_position"] = (
    inventory["current_stock"]
    + inventory["incoming_stock"]
)

# Verify the calculation
print("Inventory shape:", inventory.shape)

print(
    "Total current stock:",
    inventory["current_stock"].sum()
)

print(
    "Total incoming stock:",
    inventory["incoming_stock"].sum()
)

print(
    "Total inventory position:",
    inventory["inventory_position"].sum()
)

# Check calculation
assert (
    inventory["inventory_position"]
    == inventory["current_stock"] + inventory["incoming_stock"]
).all()

print("Inventory calculation: PASSED")

display(
    inventory[
        ["item_id", "current_stock",
         "incoming_stock", "inventory_position"]
    ].head()
)


Inventory shape: (3049, 9)
Total current stock: 167967
Total incoming stock: 45382
Total inventory position: 213349
Inventory calculation: PASSED


,item_id,current_stock,incoming_stock,inventory_position
0,HOBBIES_1_001,18,21,39
1,HOBBIES_1_002,80,25,105
2,HOBBIES_1_003,69,5,74
3,HOBBIES_1_004,49,23,72
4,HOBBIES_1_005,49,11,60



### 4. Load Historical Sales Data


In [5]:

# Load Walmart historical sales
sales_history = pd.read_csv(
    DATA_DIR / "sales_train_validation.csv"
)

# Select our store
sales_ca1 = sales_history[
    sales_history["store_id"] == "CA_1"
].copy()

# Identify daily sales columns
day_columns = sorted(
    [col for col in sales_ca1.columns
     if col.startswith("d_")],
    key=lambda col: int(col.split("_")[1])
)

# Create a product-by-day sales table
sales_matrix = sales_ca1.set_index(
    "item_id"
)[day_columns].astype("float32")

print("Sales matrix shape:", sales_matrix.shape)
print("Unique products:", sales_matrix.index.nunique())
print("Historical days:", len(day_columns))
print("Missing values:", sales_matrix.isna().sum().sum())


Sales matrix shape: (3049, 1913)
Unique products: 3049
Historical days: 1913
Missing values: 0



### 5. Load and Prepare Calendar Data


In [6]:

# Load the M5 calendar
calendar = pd.read_csv(
    DATA_DIR / "calendar.csv"
)

# Convert dates to datetime format
calendar["date"] = pd.to_datetime(
    calendar["date"]
)

# Match calendar dates to sales day columns
date_to_d = calendar.set_index("date")["d"]

# Check the calendar
print("Calendar shape:", calendar.shape)

print(
    "First sales date:",
    calendar.loc[
        calendar["d"] == day_columns[0], "date"
    ].iloc[0].date()
)

print(
    "Last sales date:",
    calendar.loc[
        calendar["d"] == day_columns[-1], "date"
    ].iloc[0].date()
)

print(
    "Missing dates:",
    calendar["date"].isna().sum()
)

print(
    "Duplicate dates:",
    calendar["date"].duplicated().sum()
)


Calendar shape: (1969, 14)
First sales date: 2011-01-29
Last sales date: 2016-04-24
Missing dates: 0
Duplicate dates: 0



### 6. Define the 28-Day Forecasting Period


In [7]:

# Choose a historical forecasting date
forecast_origin = pd.Timestamp("2016-03-27")

# Dates used to calculate historical sales features
history_dates = pd.date_range(
    end=forecast_origin,
    periods=29
)

# Dates we want to predict
future_dates = pd.date_range(
    start=forecast_origin + pd.Timedelta(days=1),
    periods=28
)

# Match dates to sales columns
history_cols = date_to_d.loc[history_dates].tolist()
future_cols = date_to_d.loc[future_dates].tolist()

# Verify dates
print("Forecast origin:", forecast_origin.date())
print("History start:", history_dates.min().date())
print("History end:", history_dates.max().date())
print("Forecast start:", future_dates.min().date())
print("Forecast end:", future_dates.max().date())

print("\nHistorical days:", len(history_cols))
print("Forecast days:", len(future_cols))



Forecast origin: 2016-03-27
History start: 2016-02-28
History end: 2016-03-27
Forecast start: 2016-03-28
Forecast end: 2016-04-24

Historical days: 29
Forecast days: 28



### 7. Calculate Historical Demand Features


In [8]:

# Get historical sales values
history_values = sales_matrix[
    history_cols
].to_numpy()

# Calculate sales features for each product
demand_features = pd.DataFrame({
    "item_id": sales_matrix.index,
    "lag_1": history_values[:, -1],
    "lag_7": history_values[:, -8],
    "lag_28": history_values[:, 0],
    "rolling_mean_7": history_values[:, -7:].mean(axis=1),
    "rolling_mean_28": history_values[:, -28:].mean(axis=1)
})

# Verify results
print("Demand features shape:", demand_features.shape)
print("Unique products:", demand_features["item_id"].nunique())
print("Missing values:", demand_features.isna().sum().sum())

display(demand_features.head())


Demand features shape: (3049, 6)
Unique products: 3049
Missing values: 0


,item_id,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28
0,HOBBIES_1_001,1.0,3.0,2.0,1.142857,1.000000
1,HOBBIES_1_002,1.0,0.0,0.0,0.571429,0.321429
2,HOBBIES_1_003,0.0,0.0,0.0,0.285714,0.535714
3,HOBBIES_1_004,6.0,1.0,6.0,3.571429,1.892857
4,HOBBIES_1_005,0.0,2.0,1.0,0.857143,1.107143



### 8. Prepare 28-Day Forecast Inputs


In [9]:

# Create 28 rows for every product
forecast_inputs = demand_features.loc[
    demand_features.index.repeat(28)
].reset_index(drop=True)

# Add forecast horizon (1 to 28)
forecast_inputs["horizon"] = np.tile(
    np.arange(1, 29),
    len(demand_features)
)

# Add future dates
forecast_inputs["target_date"] = np.tile(
    future_dates.to_numpy(),
    len(demand_features)
)

# Add calendar features
forecast_inputs["day_of_week"] = (
    forecast_inputs["target_date"].dt.dayofweek
)

forecast_inputs["month"] = (
    forecast_inputs["target_date"].dt.month
)

forecast_inputs["day_of_month"] = (
    forecast_inputs["target_date"].dt.day
)

# Verify results
print("Forecast input shape:", forecast_inputs.shape)
print("Unique products:", forecast_inputs["item_id"].nunique())
print("Forecast horizons:", forecast_inputs["horizon"].nunique())
print("Missing values:", forecast_inputs.isna().sum().sum())

display(forecast_inputs.head())


Forecast input shape: (85372, 11)
Unique products: 3049
Forecast horizons: 28
Missing values: 0


,item_id,lag_1,lag_7,lag_28,rolling_mean_7,rolling_mean_28,horizon,target_date,day_of_week,month,day_of_month
0,HOBBIES_1_001,1.0,3.0,2.0,1.142857,1.0,1,2016-03-28,0,3,28
1,HOBBIES_1_001,1.0,3.0,2.0,1.142857,1.0,2,2016-03-29,1,3,29
2,HOBBIES_1_001,1.0,3.0,2.0,1.142857,1.0,3,2016-03-30,2,3,30
3,HOBBIES_1_001,1.0,3.0,2.0,1.142857,1.0,4,2016-03-31,3,3,31
4,HOBBIES_1_001,1.0,3.0,2.0,1.142857,1.0,5,2016-04-01,4,4,1



### 9. Generate 28-Day Demand Forecasts


In [10]:

# Prepare the exact features used during training
X_forecast = forecast_inputs[
    feature_columns
].copy()

# Match the saved model's product categories
trained_categories = (
    forecast_model.booster_
    .pandas_categorical[0]
)

X_forecast["item_id"] = pd.Categorical(
    X_forecast["item_id"],
    categories=trained_categories
)

# Handle products not seen during training
X_forecast["item_id"] = (
    X_forecast["item_id"]
    .fillna("UNKNOWN_PRODUCT")
)

# Generate predictions
predictions = forecast_model.predict(
    X_forecast,
    num_iteration=forecast_model.best_iteration_
)

# Prevent negative sales predictions
forecast_inputs["predicted_sales"] = np.clip(
    predictions, 0, None
)

# Verify results
print("Total predictions:", len(predictions))
print("Missing predictions:",
      forecast_inputs["predicted_sales"].isna().sum())
print("Negative predictions:",
      (forecast_inputs["predicted_sales"] < 0).sum())

display(
    forecast_inputs[
        ["item_id", "target_date",
         "horizon", "predicted_sales"]
    ].head()
)


Total predictions: 85372
Missing predictions: 0
Negative predictions: 0


,item_id,target_date,horizon,predicted_sales
0,HOBBIES_1_001,2016-03-28,1,0.927203
1,HOBBIES_1_001,2016-03-29,2,0.887224
2,HOBBIES_1_001,2016-03-30,3,0.887224
3,HOBBIES_1_001,2016-03-31,4,0.887224
4,HOBBIES_1_001,2016-04-01,5,0.989048



### 10. Calculate Total 28-Day Forecast Demand


In [11]:

# Sum the 28 daily forecasts for each product
demand_forecast = (
    forecast_inputs
    .groupby("item_id", as_index=False)
    .agg(
        predicted_demand_28d=("predicted_sales", "sum")
    )
)

# Verify results
print("Forecast summary shape:", demand_forecast.shape)

print(
    "Unique products:",
    demand_forecast["item_id"].nunique()
)

print(
    "Total predicted units:",
    round(demand_forecast["predicted_demand_28d"].sum(), 2)
)

print(
    "Missing values:",
    demand_forecast.isna().sum().sum()
)

print(
    "Negative demand:",
    (demand_forecast["predicted_demand_28d"] < 0).sum()
)

display(demand_forecast.head())


Forecast summary shape: (3049, 2)
Unique products: 3049
Total predicted units: 127673.37
Missing values: 0
Negative demand: 0


,item_id,predicted_demand_28d
0,FOODS_1_001,23.112338
1,FOODS_1_002,14.082901
2,FOODS_1_003,22.402180
3,FOODS_1_004,64.979657
4,FOODS_1_005,37.921139



### 11. Combine Demand Forecasts with Inventory Data


In [12]:

# Combine inventory and forecast information
inventory_plan = inventory.merge(
    demand_forecast,
    on="item_id",
    how="left",
    validate="one_to_one"
)

# Verify the combined dataset
print("Inventory plan shape:", inventory_plan.shape)

print(
    "Unique products:",
    inventory_plan["item_id"].nunique()
)

print(
    "Missing forecasts:",
    inventory_plan["predicted_demand_28d"].isna().sum()
)

print(
    "Total predicted demand:",
    round(inventory_plan["predicted_demand_28d"].sum(), 2)
)

# Ensure no products were lost
assert len(inventory_plan) == len(inventory)
assert inventory_plan["predicted_demand_28d"].notna().all()

print("Inventory-forecast integration: PASSED")

display(
    inventory_plan[
        [
            "item_id",
            "current_stock",
            "incoming_stock",
            "inventory_position",
            "lead_time_days",
            "safety_stock",
            "predicted_demand_28d"
        ]
    ].head()
)


Inventory plan shape: (3049, 10)
Unique products: 3049
Missing forecasts: 0
Total predicted demand: 127673.37
Inventory-forecast integration: PASSED


,item_id,current_stock,incoming_stock,inventory_position,lead_time_days,safety_stock,predicted_demand_28d
0,HOBBIES_1_001,18,21,39,12,12,28.225559
1,HOBBIES_1_002,80,25,105,5,14,11.992657
2,HOBBIES_1_003,69,5,74,5,22,14.924807
3,HOBBIES_1_004,49,23,72,13,22,64.371722
4,HOBBIES_1_005,49,11,60,8,9,28.000766



### 12. Calculate Inventory Reorder Points


In [13]:

# Estimate average daily demand
inventory_plan["avg_daily_demand"] = (
    inventory_plan["predicted_demand_28d"] / 28
)

# Estimate demand during supplier lead time
inventory_plan["lead_time_demand"] = (
    inventory_plan["avg_daily_demand"]
    * inventory_plan["lead_time_days"]
)

# Calculate reorder point
inventory_plan["reorder_point"] = np.ceil(
    inventory_plan["lead_time_demand"]
    + inventory_plan["safety_stock"]
).astype(int)

# Check whether each product has reached its reorder point
inventory_plan["reorder_needed"] = (
    inventory_plan["inventory_position"]
    <= inventory_plan["reorder_point"]
)

# Display results
print("Total products:", len(inventory_plan))

print(
    "Products needing reorder:",
    inventory_plan["reorder_needed"].sum()
)

print(
    "Products not needing reorder:",
    (~inventory_plan["reorder_needed"]).sum()
)

print(
    "Missing values:",
    inventory_plan[
        ["avg_daily_demand", "lead_time_demand",
         "reorder_point", "reorder_needed"]
    ].isna().sum().sum()
)

display(
    inventory_plan[
        ["item_id", "inventory_position",
         "lead_time_days", "reorder_point",
         "reorder_needed"]
    ].head()
)


Total products: 3049
Products needing reorder: 273
Products not needing reorder: 2776
Missing values: 0


,item_id,inventory_position,lead_time_days,reorder_point,reorder_needed
0,HOBBIES_1_001,39,12,25,False
1,HOBBIES_1_002,105,5,17,False
2,HOBBIES_1_003,74,5,25,False
3,HOBBIES_1_004,72,13,52,False
4,HOBBIES_1_005,60,8,18,False



### 13. Calculate Recommended Reorder Quantities


In [14]:

# Extra days of stock to cover after delivery
review_period_days = 7

# Calculate how many days of demand we want to cover
inventory_plan["coverage_days"] = (
    inventory_plan["lead_time_days"] + review_period_days
)

# Calculate target inventory level
inventory_plan["target_stock"] = np.ceil(
    inventory_plan["avg_daily_demand"]
    * inventory_plan["coverage_days"]
    + inventory_plan["safety_stock"]
).astype(int)

# Calculate proposed order quantities
inventory_plan["recommended_order_qty"] = np.where(
    inventory_plan["reorder_needed"],
    np.maximum(
        inventory_plan["target_stock"]
        - inventory_plan["inventory_position"],
        0
    ),
    0
).astype(int)

# Verify results
print(
    "Products with recommended orders:",
    (inventory_plan["recommended_order_qty"] > 0).sum()
)

print(
    "Total recommended units:",
    inventory_plan["recommended_order_qty"].sum()
)

print(
    "Negative order quantities:",
    (inventory_plan["recommended_order_qty"] < 0).sum()
)

print(
    "Missing values:",
    inventory_plan["recommended_order_qty"].isna().sum()
)

display(
    inventory_plan.loc[
        inventory_plan["recommended_order_qty"] > 0,
        [
            "item_id",
            "inventory_position",
            "reorder_point",
            "target_stock",
            "recommended_order_qty"
        ]
    ].head(10)
)


Products with recommended orders: 273
Total recommended units: 19045
Negative order quantities: 0
Missing values: 0


,item_id,inventory_position,reorder_point,target_stock,recommended_order_qty
7,HOBBIES_1_008,97,97,148,51
18,HOBBIES_1_019,107,112,160,53
29,HOBBIES_1_030,56,82,123,67
35,HOBBIES_1_036,25,27,31,6
44,HOBBIES_1_045,23,29,32,9
47,HOBBIES_1_048,96,153,237,141
87,HOBBIES_1_090,56,67,95,39
103,HOBBIES_1_108,16,36,44,28
142,HOBBIES_1_147,56,62,98,42
146,HOBBIES_1_151,18,31,35,17



### 14. Prioritize Inventory Reorder Recommendations


In [15]:

# Assign inventory priority
inventory_plan["priority"] = np.select(
    [
        inventory_plan["reorder_needed"] &
        (
            inventory_plan["inventory_position"]
            < inventory_plan["reorder_point"] * 0.5
        ),

        inventory_plan["reorder_needed"]
    ],
    [
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

# Count products by priority
print("Priority distribution:")
print(inventory_plan["priority"].value_counts())

# Show the most urgent recommendations
priority_order = {
    "HIGH": 0,
    "MEDIUM": 1,
    "LOW": 2
}

reorder_recommendations = inventory_plan.loc[
    inventory_plan["recommended_order_qty"] > 0
].copy()

reorder_recommendations["priority_rank"] = (
    reorder_recommendations["priority"].map(priority_order)
)

reorder_recommendations = (
    reorder_recommendations
    .sort_values(
        ["priority_rank", "recommended_order_qty"],
        ascending=[True, False]
    )
    .drop(columns="priority_rank")
)

display(
    reorder_recommendations[
        [
            "item_id",
            "cat_id",
            "inventory_position",
            "reorder_point",
            "recommended_order_qty",
            "priority"
        ]
    ].head(10)
)

print(
    "\nTotal prioritized reorder products:",
    len(reorder_recommendations)
)


Priority distribution:
priority
LOW       2776
MEDIUM     211
HIGH        62
Name: count, dtype: int64


,item_id,cat_id,inventory_position,reorder_point,recommended_order_qty,priority
2314,FOODS_3_090,FOODS,56,339,744,HIGH
2476,FOODS_3_252,FOODS,60,532,735,HIGH
2344,FOODS_3_120,FOODS,88,336,632,HIGH
2810,FOODS_3_586,FOODS,77,257,470,HIGH
2811,FOODS_3_587,FOODS,51,257,402,HIGH
2288,FOODS_3_064,FOODS,112,254,302,HIGH
1846,FOODS_2_019,FOODS,58,240,289,HIGH
2946,FOODS_3_723,FOODS,69,236,278,HIGH
2735,FOODS_3_511,FOODS,39,181,268,HIGH
2519,FOODS_3_295,FOODS,109,254,266,HIGH



Total prioritized reorder products: 273



### 15. Generate Explainable Reorder Recommendations


In [16]:

# Create clear explanations for each recommended order
def explain_reorder(row):
    return (
        f"{row['priority']} priority: "
        f"Inventory position is {row['inventory_position']} units, "
        f"below or at the reorder point of {row['reorder_point']} units. "
        f"Recommend ordering {row['recommended_order_qty']} units "
        f"to reach the target stock level of {row['target_stock']} units."
    )

# Add an explanation to each reorder recommendation
reorder_recommendations["reason"] = (
    reorder_recommendations.apply(explain_reorder, axis=1)
)

# Mark all recommendations as awaiting human review
reorder_recommendations["approval_status"] = "PENDING"

# Verify results
print("Total recommendations:", len(reorder_recommendations))

print(
    "Missing explanations:",
    reorder_recommendations["reason"].isna().sum()
)

print(
    "Approval status:",
    reorder_recommendations["approval_status"].value_counts()
)

display(
    reorder_recommendations[
        ["item_id", "priority",
         "recommended_order_qty", "reason",
         "approval_status"]
    ].head(5)
)


Total recommendations: 273
Missing explanations: 0
Approval status: approval_status
PENDING    273
Name: count, dtype: int64


,item_id,priority,recommended_order_qty,reason,approval_status
2314,FOODS_3_090,HIGH,744,"HIGH priority: Inventory position is 56 units,...",PENDING
2476,FOODS_3_252,HIGH,735,"HIGH priority: Inventory position is 60 units,...",PENDING
2344,FOODS_3_120,HIGH,632,"HIGH priority: Inventory position is 88 units,...",PENDING
2810,FOODS_3_586,HIGH,470,"HIGH priority: Inventory position is 77 units,...",PENDING
2811,FOODS_3_587,HIGH,402,"HIGH priority: Inventory position is 51 units,...",PENDING



### 16. Simulate Human Approval of Reorder Recommendations


In [17]:

# Create a separate copy for approval testing
approval_demo = reorder_recommendations.copy()

# Select the first recommended product
selected_item = approval_demo.iloc[0]["item_id"]

# Simulate a manager approving this recommendation
approval_demo.loc[
    approval_demo["item_id"] == selected_item,
    "approval_status"
] = "APPROVED"

# Verify approval results
print("Selected product:", selected_item)

print("\nApproval status distribution:")
print(approval_demo["approval_status"].value_counts())

print("\nApproved recommendation:")
display(
    approval_demo.loc[
        approval_demo["approval_status"] == "APPROVED",
        [
            "item_id",
            "recommended_order_qty",
            "priority",
            "approval_status"
        ]
    ]
)

# Original recommendations remain unchanged
assert (
    reorder_recommendations["approval_status"] == "PENDING"
).all()

print("Original recommendations unchanged: PASSED")


Selected product: FOODS_3_090

Approval status distribution:
approval_status
PENDING     272
APPROVED      1
Name: count, dtype: int64

Approved recommendation:


,item_id,recommended_order_qty,priority,approval_status
2314,FOODS_3_090,744,HIGH,APPROVED


Original recommendations unchanged: PASSED



### 17. Create a Simulated Approval Audit Log


In [18]:

# Find the approved recommendation
approved_item = approval_demo.loc[
    approval_demo["approval_status"] == "APPROVED"
].iloc[0]

# Create a simulated approval record
approval_log = pd.DataFrame([{
    "item_id": approved_item["item_id"],
    "recommended_order_qty": int(
        approved_item["recommended_order_qty"]
    ),
    "decision": "APPROVED",
    "approved_by": "DEMO_MANAGER",
    "decision_time": pd.Timestamp.now(),
    "simulation": True
}])

# Verify the record
print("Approval records:", len(approval_log))
print("Simulated records:", approval_log["simulation"].sum())
print("Missing values:", approval_log.isna().sum().sum())

display(approval_log)


Approval records: 1
Simulated records: 1
Missing values: 0


,item_id,recommended_order_qty,decision,approved_by,decision_time,simulation
0,FOODS_3_090,744,APPROVED,DEMO_MANAGER,2026-11-01 11:44:40.022620,True



### 18. Export Inventory Recommendations and Approval Log


In [19]:

# Create an output folder
output_dir = DATA_DIR / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# File locations
recommendations_path = (
    output_dir / "inventory_recommendations.csv"
)

approval_log_path = (
    output_dir / "approval_log_demo.csv"
)

# Save original pending recommendations
reorder_recommendations.to_csv(
    recommendations_path,
    index=False
)

# Save simulated approval history
approval_log.to_csv(
    approval_log_path,
    index=False
)

# Check saved files
saved_recommendations = pd.read_csv(
    recommendations_path
)

saved_approvals = pd.read_csv(
    approval_log_path
)

print(
    "Saved recommendations:",
    len(saved_recommendations)
)

print(
    "Saved approval records:",
    len(saved_approvals)
)

print(
    "Pending recommendations:",
    (saved_recommendations["approval_status"] == "PENDING").sum()
)

print(
    "Files saved successfully:",
    recommendations_path.exists()
    and approval_log_path.exists()
)


Saved recommendations: 273
Saved approval records: 1
Pending recommendations: 273
Files saved successfully: True



## Conclusion

Developed an inventory replenishment system integrating the trained 28-day LightGBM demand forecasting model with simulated inventory data for 3,049 products.

### Key Results
- **Total predicted demand:** 127,673.37 units
- **Products requiring replenishment:** 273
- **Total recommended reorder quantity:** 19,045 units
- **HIGH priority:** 62 products
- **MEDIUM priority:** 211 products
- **LOW priority:** 2,776 products

Implemented reorder-point calculations, replenishment recommendations, priority classification, explainable decisions, and a simulated human approval workflow.

Inventory recommendations and a demonstration approval log were successfully exported for downstream analysis.

**Limitations:** Inventory quantities and supplier parameters are simulated. Replenishment policies are simplified, forecast uncertainty remains significant, and no real purchase orders were executed.

**Next:** Business Decision and Risk Analysis (Notebook 05).
